<div style="background:linear-gradient(120deg,#0B1F3A,#123A4C);color:#F6F1E7;padding:38px;border-radius:18px;">
<p style="color:#F4C95D;letter-spacing:.16em;font-size:12px;">PLAYGROUND S6E9 · PRIVATE-FIRST META LAB</p>
<h1 style="color:#F6F1E7;font-size:38px;line-height:1.12;">0.94656 Is Not a Model</h1>
<p style="font-size:19px;">Fifteen honest OOF views reach 0.946426 without reading the public split.</p>
<p style="color:#00B4A6;">Den Pugovkin · optimise signal, not the visible fifth</p></div>

### TL;DR

- Our previous six-model L1 stack scored **0.946350 OOF / 0.94646 public**.
- Fresh `0.94656` notebooks move **57,314 test rows** in four bands chosen from public-LB feedback. Their author says the change helps the public 20% and has no expected private gain. We reject it.
- The new candidate adds six independent feature views plus two RealMLP rankings from the public [six-view OOF library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library).
- Full local verification: **0.946425550 OOF**, gain **+0.000075977**, and **5/5 folds** above the six-model control.
- `submission.csv` switches to the expanded stack only if the same +0.000020 / 4-of-5 gate passes in this run.

**Run:** attach the competition, [S6E9 OOF](https://www.kaggle.com/datasets/najiama/s6e9-oof), and [S6E9 Six Feature Views OOF Library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library). CPU, Internet Off.

[01 Fresh audit](#sources) · [02 Input contract](#audit) · [03 New signal](#diversity) · [04 Fold-honest stack](#stack) · [05 Decision](#decision)

<a id="sources"></a>
## 01 · The new 0.94656 wave: model gain or public feedback?

| Fresh source checked | What it actually does | Decision |
|---|---|---|
| [Reading the Public Split](https://www.kaggle.com/code/megayak/s6e9-0-94656-reading-the-public-split) | Starts from the 0.94651 Zoom Zoom file and swaps four rank bands. Two successful bands were selected from 15 public submissions. It changes 57,314 ranks. | Useful leaderboard anatomy; reject for the private solution. |
| [Does Breaking Ties Help?](https://www.kaggle.com/code/chinzorigtganbat/s6e9-does-breaking-ties-help) | Reproduces those same four bands. The author clearly labels the candidate as public-score evidence, not a better model. | No new predictive signal. |
| [Grandmaster SOTA Meta-Blend](https://www.kaggle.com/code/amanatar/s6e9-grandmaster-sota-meta-blend) | Reuses the public-calibrated bands, adds 2% public-file consensus and four hard boundary shifts. | Reject from the official path: components are selected without OOF evidence. |
| [Six Feature Views OOF Library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library) | Six leak-free feature pipelines on 10 folds, plus 10-fold and 3-seed RealMLP OOF/test predictions. | Accept as the one material change. |
| [Complete HiRGE OOF](https://www.kaggle.com/datasets/tamerlanomralinov/s6e9-complete-oof-npy-hirge) | Many diverse neural/tree rankings, but its best tested addition produced 0.946396 in our audit. | Exclude: the six-view + RealMLP pool reached 0.946426 without it. |

The public-band method is unusually honest about its limitation: the four swaps were retained because this particular public sample liked them. That is selection on the evaluation set. Our notebook uses only row-level OOF evidence to choose the official file.

In [ ]:
from pathlib import Path
import hashlib, json, os, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import HTML, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED, ID, TARGET = 42, "id", "Will_Buy_EV"
NAVY, TEAL, GOLD, CORAL, PAPER = "#0B1F3A", "#00B4A6", "#F4C95D", "#E85D4C", "#F6F1E7"
STYLE = dict(template="plotly_white", font=dict(family="Inter, Arial", color=NAVY),
             title_font=dict(color=NAVY), colorway=[TEAL, GOLD, CORAL, "#526D82"])
ROOT = Path(os.environ.get("S6E9_INPUT_ROOT", "/kaggle/input" if Path("/kaggle/input").exists() else Path.cwd()))
OUTPUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/oof-stack")
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Input root: {ROOT}\nOutput: {OUTPUT}")

In [ ]:
def one(pattern: str, roots=(ROOT,), contains: str | None = None) -> Path:
    hits = sorted({p for root in roots for p in root.rglob(pattern) if p.is_file()})
    if contains:
        filtered = [p for p in hits if contains.lower() in str(p).lower()]
        if filtered:
            hits = filtered
    if len(hits) != 1:
        raise RuntimeError(f"Expected one {pattern}; found {len(hits)}: {hits[:8]}")
    return hits[0]

train_path = one("train.csv", contains="playground-series-s6e9")
sample_path = one("sample_submission.csv", contains="playground-series-s6e9")
test_path = one("test.csv", contains="playground-series-s6e9")
train, sample, test = pd.read_csv(train_path), pd.read_csv(sample_path), pd.read_csv(test_path)
test = pd.DataFrame({ID: sample[ID]}).merge(test, on=ID, how="left", validate="one_to_one")
target_values = set(train[TARGET].dropna().unique())
y = train[TARGET].map({"No": 0, "Yes": 1}) if target_values <= {"No", "Yes"} else train[TARGET].astype(int)
assert train[ID].is_unique and sample[ID].is_unique and test[ID].equals(sample[ID]) and y.isin([0, 1]).all()
folds = np.empty(len(train), dtype=np.int8)
for fold, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, y)):
    folds[valid] = fold
fingerprint = hashlib.sha256(pd.DataFrame({ID: train[ID], "fold": folds}).to_csv(index=False).encode()).hexdigest()
print(train.shape, sample.shape, f"positive rate={y.mean():.3%}", f"fold fingerprint={fingerprint}")

<a id="audit"></a>
## 02 · Three input contracts, no filename guessing

The core six files must align one-to-one with competition IDs. The new library must contain both wide OOF/test tables and the RealMLP pair. We verify target labels, row order, ID uniqueness, finiteness, and the library's 10-fold assignment. Any missing or changed source stops the notebook instead of silently producing a different ensemble.

In [ ]:
CORE = {
    "01 Blend": ("01_blend_oof.csv", "01_submission.csv"),
    "Pure LGBM v1": ("Pure LGBM_V1_oof.csv", "Pure LGBM_V1_test.csv"),
    "Pure LGBM v3": ("Pure LGBM_V3_oof.csv", "Pure LGBM_V3_test.csv"),
    "Sergey LGBM": ("Sergey_LGBM_oof.csv", "Sergey_LGBM_submission.csv"),
    "XGB triple TE 10f": ("XGBoost_Triple_TE_10folds_oof.csv", "XGBoost_Triple_TE_10folds_test.csv"),
    "XGB triple TE 5f": ("XGBoost_Triple_TE_5folds_oof.csv", "XGBoost_Triple_TE_5folds_test.csv"),
}

def prediction(path: Path, ids: pd.Series, label: str) -> np.ndarray:
    frame = pd.read_csv(path)
    assert ID in frame and TARGET not in frame.columns[:-1], f"Unsafe schema: {path}"
    pred_cols = [c for c in frame if c != ID]
    assert len(pred_cols) == 1 and frame[ID].is_unique, f"Ambiguous schema: {path}"
    aligned = pd.DataFrame({ID: ids}).merge(frame[[ID, pred_cols[0]]], on=ID, how="left", validate="one_to_one")
    values = aligned[pred_cols[0]].to_numpy(float)
    assert len(values) == len(ids) and np.isfinite(values).all() and np.std(values) > 0
    print(f"✓ {label:<20} {path.name}")
    return values

oof, test_pred = {}, {}
for name, (oof_name, test_name) in CORE.items():
    oof[name] = prediction(one(oof_name), train[ID], name + " OOF")
    test_pred[name] = prediction(one(test_name), sample[ID], name + " test")

In [ ]:
def rank01(values: np.ndarray) -> np.ndarray:
    return pd.Series(values).rank(method="average", pct=True).to_numpy()


def aligned(frame: pd.DataFrame, ids: pd.Series, column: str) -> np.ndarray:
    assert ID in frame and frame[ID].is_unique and column in frame
    out = pd.DataFrame({ID: ids}).merge(frame[[ID, column]], on=ID, how="left", validate="one_to_one")
    values = out[column].to_numpy(float)
    assert len(values) == len(ids) and np.isfinite(values).all() and np.std(values) > 0
    return values


SIX_COLUMNS = [
    "A_lgbm_triple_te_digits_3seed", "B_xgb_on_A_features",
    "C_no_digits_windows_lift_sm2_30_300", "D_no_exact_key_ladder_windows",
    "E_ladder25_250_2500_lift_sm5_50_500", "F_exact_rate_as_init_score",
    "ensemble",
]
REALMLP_COLUMNS = ["G_realmlp_10fold", "G_realmlp_3seed"]

In [ ]:
six_oof_path = one("oof_six_views.csv")
six_test_path = one("test_six_views.csv")
real_oof_path = one("oof_realmlp_g.csv")
real_test_path = one("test_realmlp_g.csv")

six_oof, six_test = pd.read_csv(six_oof_path), pd.read_csv(six_test_path)
real_oof, real_test = pd.read_csv(real_oof_path), pd.read_csv(real_test_path)
assert six_oof[TARGET].astype(int).to_numpy().tolist() == y.astype(int).to_numpy().tolist()
assert real_oof[TARGET].astype(int).to_numpy().tolist() == y.astype(int).to_numpy().tolist()
assert set(six_oof["fold"].unique()) == set(range(10))
assert np.array_equal(six_oof["fold"].to_numpy(), real_oof["fold"].to_numpy())

for column in SIX_COLUMNS:
    oof[f"view/{column}"] = aligned(six_oof, train[ID], column)
    test_pred[f"view/{column}"] = aligned(six_test, sample[ID], column)
for column in REALMLP_COLUMNS:
    oof[f"view/{column}"] = aligned(real_oof, train[ID], column)
    test_pred[f"view/{column}"] = aligned(real_test, sample[ID], column)
print(f"Loaded {len(SIX_COLUMNS)} feature-view rankings + {len(REALMLP_COLUMNS)} RealMLP rankings")

In [ ]:
EXTRA_NAMES = [name for name in oof if name not in CORE]
audit = pd.DataFrame({
    "Model": list(oof),
    "OOF AUC": [roc_auc_score(y, values) for values in oof.values()],
    "Pool": ["core6" if name in CORE else "new view" for name in oof],
})
assert len(EXTRA_NAMES) == 9 and len(oof) == 15
display(audit.sort_values("OOF AUC", ascending=False).style
        .format({"OOF AUC": "{:.7f}"})
        .background_gradient(subset=["OOF AUC"], cmap="YlGnBu"))
print("New members:", EXTRA_NAMES)

<a id="diversity"></a>
## 03 · Feature engineering changes the errors, not just the score

The six new views deliberately change how the synthetic numeric structure is represented:

| View | Representation change | Why it can complement the core |
|---|---|---|
| A/B | Triple target encoding + digits, learned by LightGBM/XGBoost | Strong high-resolution generator signal |
| C | No digits; centred income and commute windows | Smooth local structure rather than decimal artifacts |
| D | Removes the exact-income key; multiscale integer ladder | Reduces dependence on memorised exact values |
| E | A different 25/250/2500 income ladder | Shifts discretisation boundaries |
| F | Exact-income rate enters as `init_score` | Uses the rate as a prior instead of another split feature |
| G | RealMLP, three seeds | Different learner and the lowest correlation among the strong new views |

Standalone AUC is only half of the question. The heatmap shows rank correlation; the meta-model decides whether the residual disagreement predicts the target. All predictions are converted to percentile ranks because ROC AUC depends only on ordering.

In [ ]:
rank_oof = pd.DataFrame({name: rank01(values) for name, values in oof.items()})
rank_test = pd.DataFrame({name: rank01(test_pred[name]) for name in rank_oof})
corr = rank_oof.corr(method="spearman")
fig = px.imshow(corr, zmin=.85, zmax=1, color_continuous_scale=[[0, CORAL], [.55, GOLD], [1, TEAL]],
                text_auto=".3f", aspect="auto", title="Prediction-rank correlation: quality is not diversity")
fig.update_layout(**STYLE, height=640)
fig.show()

<a id="stack"></a>
## 04 · Same regularisation, nine genuinely new rankings

The control is the exact six-source `C=1` L1 recipe that produced **0.946350 OOF / 0.94646 public**. The candidate changes only the input pool: nine OOF/test columns from the six-view library are added. For each frozen outer fold, scaler and L1 logistic regression fit on the other four folds and predict the held fold once. Test predictions are averaged across five meta-models.

The candidate is accepted only with overall gain above **0.000020** and at least **4/5** positive fold deltas. `C` is inherited from the control; there is no regularisation sweep on the displayed OOF result.

In [ ]:
def store(oof_hat, test_hat, scores, weight_rows):
    assert np.isfinite(oof_hat).all() and np.isfinite(test_hat).all()
    return oof_hat, test_hat, scores, pd.DataFrame(weight_rows)


def fit_l1(columns, recipe):
    X, Xtest = rank_oof[columns].to_numpy(), rank_test[columns].to_numpy()
    oof_hat, test_hat, scores, rows = np.zeros(len(train)), np.zeros(len(sample)), [], []
    for fold in range(5):
        fit, valid = folds != fold, folds == fold
        scaler = StandardScaler().fit(X[fit])
        model = LogisticRegression(penalty="l1", solver="saga", C=1.0, max_iter=1200,
                                   tol=1e-5, random_state=SEED + fold, n_jobs=-1)
        model.fit(scaler.transform(X[fit]), y[fit])
        oof_hat[valid] = model.predict_proba(scaler.transform(X[valid]))[:, 1]
        test_hat += model.predict_proba(scaler.transform(Xtest))[:, 1] / 5
        scores.append(roc_auc_score(y[valid], oof_hat[valid]))
        rows.append(dict(recipe=recipe, fold=fold, **dict(zip(columns, model.coef_[0]))))
        print(f"{recipe:<16} fold {fold}: {scores[-1]:.7f}")
    return store(oof_hat, test_hat, scores, rows)

In [ ]:
def py(value):
    if isinstance(value, dict): return {k: py(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)): return [py(v) for v in value]
    if isinstance(value, (np.bool_, bool)): return bool(value)
    if isinstance(value, np.integer): return int(value)
    if isinstance(value, (np.floating, float)): return float(value)
    return value


def compare(parent, child):
    parent_auc, child_auc = roc_auc_score(y, parent[0]), roc_auc_score(y, child[0])
    fold_deltas = np.asarray(child[2]) - np.asarray(parent[2])
    wins, delta = int((fold_deltas > 0).sum()), float(child_auc - parent_auc)
    return {"auc": float(child_auc), "delta": delta, "fold_wins": wins,
            "fold_deltas": fold_deltas.tolist(),
            "accepted": bool(delta > 0.000020 and wins >= 4)}


started = time.perf_counter()
results = {"l1_core6": fit_l1(list(CORE), "l1_core6")}
results["l1_core6_views"] = fit_l1(list(oof), "l1_core6_views")
candidate = compare(results["l1_core6"], results["l1_core6_views"])
selected = "l1_core6_views" if candidate["accepted"] else "l1_core6"
decision = {"parent": "l1_core6", "candidate": candidate, "selected": selected,
            "selected_auc": float(roc_auc_score(y, results[selected][0])), "extras": EXTRA_NAMES}
print(f"Meta training: {time.perf_counter() - started:.1f} seconds")
print(json.dumps(py(decision), indent=2))

<a id="decision"></a>
## 05 · Did the new views earn the fifth decimal?

Overall gain measures the full ranking; fold deltas test repeatability. The coefficient table shows which representations remain useful after conditioning on the other fourteen predictions. Negative coefficients are allowed: with highly correlated rankings, L1 can use one member as a residual correction rather than a standalone vote.

The local reference run reached **0.946425550**, a gain of **+0.000075977**, with positive deltas in all five folds. The cells below recompute that evidence on Kaggle before writing the selected submission.

In [ ]:
rows = []
for recipe, (pred, _, scores, _) in results.items():
    for fold, score in enumerate(scores): rows.append({"Recipe": recipe, "Fold": fold, "AUC": score})
score_table = pd.DataFrame(rows)
fig = px.line(score_table, x="Fold", y="AUC", color="Recipe", markers=True,
              title="Same held rows, same folds: six-source control vs fifteen-view stack")
fig.update_layout(**STYLE)
fig.show()

coef = pd.concat([value[3] for name, value in results.items() if name != "dgp_rules"], ignore_index=True)
long_coef = coef.melt(id_vars=["recipe", "fold"], var_name="Model", value_name="Coefficient")
coef_summary = long_coef.groupby(["recipe", "Model"]).Coefficient.agg(["mean", "std", "min", "max"]).reset_index()
display(coef_summary.style.format({c: "{:+.4f}" for c in ["mean", "std", "min", "max"]}))

In [ ]:
for recipe, (meta_oof, meta_test, fold_scores, _) in results.items():
    pd.DataFrame({ID: train[ID], "fold": folds, "prediction": meta_oof}).to_csv(OUTPUT / f"oof_{recipe}.csv", index=False)
    pd.DataFrame({ID: sample[ID], TARGET: meta_test}).to_csv(OUTPUT / f"submission_{recipe}.csv", index=False)

chosen_oof, chosen_test = results[selected][0], results[selected][1]
if not ((chosen_test >= 0) & (chosen_test <= 1)).all():
    chosen_test = rank01(chosen_test)
submission = pd.DataFrame({ID: sample[ID], TARGET: chosen_test})
assert submission.shape == sample.shape and submission[ID].equals(sample[ID])
assert submission[TARGET].between(0, 1).all() and np.isfinite(submission[TARGET]).all()
submission.to_csv(OUTPUT / "submission.csv", index=False)
coef.to_csv(OUTPUT / "meta_coefficients.csv", index=False)
report = py({**decision, "selected_oof_auc": float(roc_auc_score(y, chosen_oof)),
          "per_fold_auc": [float(s) for s in results[selected][2]],
          "rows": {"train": int(len(train)), "test": int(len(sample))},
          "fold_fingerprint": fingerprint, "members": list(rank_oof),
          "extras": EXTRA_NAMES, "seed": SEED, "C": 1.0, "public_band_swaps_used": False})
(OUTPUT / "stack_report.json").write_text(json.dumps(report, indent=2))

color = TEAL if selected != "l1_core6" else GOLD
display(HTML(f'''<div style="background:{NAVY};color:{PAPER};padding:24px;border-radius:16px;">
<p style="color:{color};font-size:13px;letter-spacing:.12em;">SELECTED RECIPE</p>
<h2 style="color:{PAPER};margin:4px 0;">{selected}</h2>
<b>OOF AUC {report['selected_oof_auc']:.7f}</b><br>
submission.csv is ready · public score is not claimed until submission
</div>'''))
print(submission.head(), "\n", sorted(p.name for p in OUTPUT.iterdir()))

### What survived the audit

- The new gain comes from **feature-view diversity**, especially the RealMLP and alternative income representations. It is measured on labels that each base model did not train on.
- The expanded stack improved every frozen fold and cleared the predeclared threshold. This is stronger evidence than a one-digit public movement.
- HiRGE was inspected and tested, then left out: its expanded candidate reached about **0.946396**, below this cleaner pool's **0.946426**.
- The four public-calibrated band swaps are never applied. A public score above the model does not make those swaps useful for the private 80%.

`submission.csv` is the fifteen-view candidate only when the gate passes. The notebook also saves both recipe submissions, both OOF files, fold coefficients, and `stack_report.json`. Public score remains unclaimed until an actual submission is measured.